# Model Selection

In diesem Notebook werden Modelle anhand des Trainingsdatensatzes November 2025 bis Juni 2026 trainiert und auf dem Validierungsdatensatz Juli 2026 bewertet. Der Testdatensatz August 2026 bleibt bis zur finalen Evaluation unangetastet.

Die Datensplits stammen aus der aktualisierten temporalen Split-Logik: Fahrten, deren `concrete_ride_id` über eine Split-Grenze läuft, wurden vor der Feature-Erstellung vollständig aus Training, Validation und Test entfernt. Diese Boundary-Rides werden separat in `excluded_boundary_rides.parquet` dokumentiert und nicht für Training oder Bewertung verwendet.

Die Modellierung verwendet ausschließlich externe bzw. exogene Merkmale aus den Bereichen Zeit, Kalender und Wetter. Strukturelle Bahnmerkmale werden nicht als Modellfeatures verwendet. R² dient als zentrale Vergleichskennzahl; MAE und RMSE werden ergänzend berichtet.

## 1. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from pygam import LinearGAM, s, f, l
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


PROJECT_DIR = Path.cwd()

while not (PROJECT_DIR / "datasets").exists():
    if PROJECT_DIR == PROJECT_DIR.parent:
        raise FileNotFoundError("Could not find project directory with datasets/.")
    PROJECT_DIR = PROJECT_DIR.parent

FEATURE_DIR = PROJECT_DIR / "datasets" / "modeling" / "features"
TRAIN_PATH = FEATURE_DIR / "train_features.parquet"
VALIDATION_PATH = FEATURE_DIR / "validation_features.parquet"

TRAIN_PERIOD = "November 2025 bis Juni 2026"
VALIDATION_PERIOD = "Juli 2026"
TEST_PERIOD = "August 2026"

## 2. Daten laden


In [2]:
train = pd.read_parquet(TRAIN_PATH)
validation = pd.read_parquet(VALIDATION_PATH)

external_features = [
    "event_hour",
    "event_weekday",
    "is_public_holiday",
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
    "rr_precipitation_indicator",
    "rr_precipitation_form",
]

y_train = train["delay_target"]
y_validation = validation["delay_target"]

X_train = train[external_features].copy()
X_validation = validation[external_features].copy()

dataset_overview = pd.DataFrame(
    {
        "Dataset": ["Training", "Validation"],
        "Period": [TRAIN_PERIOD, VALIDATION_PERIOD],
        "Observations": [len(train), len(validation)],
        "Features": [X_train.shape[1], X_validation.shape[1]],
    }
)

dataset_overview

,Dataset,Period,Observations,Features
0,Training,November 2025 bis Juni 2026,689684,12
1,Validation,Juli 2026,73867,12


## 3. Datenübersicht


In [3]:
used_features = X_train.columns.tolist()

feature_overview = pd.DataFrame(
    {
        "Feature": used_features,
        "Datentyp": [X_train[column].dtype for column in used_features],
        "Missing Training": [X_train[column].isna().sum() for column in used_features],
        "Missing Validation": [X_validation[column].isna().sum() for column in used_features],
    }
)

feature_overview


,Feature,Datentyp,Missing Training,Missing Validation
0,event_hour,int32,0,0
1,event_weekday,int32,0,0
2,is_public_holiday,bool,0,0
3,ff_wind_speed_ms,float64,4806,0
4,wind_direction_sin,float64,4233,0
5,wind_direction_cos,float64,4233,0
6,fx_wind_gust_ms,float64,4806,0
7,tu_temperature_c,float64,1494,0
8,tu_relative_humidity_pct,float64,1575,0
9,rr_precipitation_mm,float64,326,0


## 4. Baselines

Die Median-Baseline dient als robuste Referenz für die typische Prognosegüte, insbesondere mit Blick auf den MAE. Die Mean-Baseline wird zusätzlich verwendet, da der Mittelwert die natürliche Referenz des R² darstellt. Beide Baselines werden ausschließlich anhand der Trainingsdaten November 2025 bis Juni 2026 bestimmt. Boundary-Rides sind in diesem Trainingsdatensatz bereits entfernt.

In [4]:
X_train_dummy = np.zeros((len(train), 1))
X_validation_dummy = np.zeros((len(validation), 1))

baseline_model = DummyRegressor(strategy="median")
baseline_model.fit(X_train_dummy, y_train)
baseline_predictions = baseline_model.predict(X_validation_dummy)

baseline_mae = mean_absolute_error(y_validation, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_validation, baseline_predictions))
baseline_r2 = r2_score(y_validation, baseline_predictions)
baseline_prediction = float(baseline_model.constant_.ravel()[0])

mean_baseline_model = DummyRegressor(strategy="mean")
mean_baseline_model.fit(X_train_dummy, y_train)
mean_baseline_predictions = mean_baseline_model.predict(X_validation_dummy)

mean_baseline_mae = mean_absolute_error(y_validation, mean_baseline_predictions)
mean_baseline_rmse = np.sqrt(mean_squared_error(y_validation, mean_baseline_predictions))
mean_baseline_r2 = r2_score(y_validation, mean_baseline_predictions)
mean_baseline_prediction = float(mean_baseline_model.constant_.ravel()[0])

print("Median Baseline:")
print(f"Prediction: {baseline_prediction:.3f}")
print(f"MAE: {baseline_mae:.3f}")
print(f"RMSE: {baseline_rmse:.3f}")
print(f"R2: {baseline_r2:.3f}")
print()
print("Mean Baseline:")
print(f"Prediction: {mean_baseline_prediction:.3f}")
print(f"MAE: {mean_baseline_mae:.3f}")
print(f"RMSE: {mean_baseline_rmse:.3f}")
print(f"R2: {mean_baseline_r2:.3f}")


Median Baseline:
Prediction: 2.000
MAE: 5.628
RMSE: 11.163
R2: -0.174

Mean Baseline:
Prediction: 5.431
MAE: 6.197
RMSE: 10.340
R2: -0.007


## 5. Feature-Gruppen


In [5]:
categorical_features = [
    "event_hour",
    "event_weekday",
    "rr_precipitation_form",
]

numeric_features = [
    "is_public_holiday",
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
    "rr_precipitation_indicator",
]

model_features = categorical_features + numeric_features

X_train_model = X_train[model_features].copy()
X_validation_model = X_validation[model_features].copy()

for column in categorical_features:
    X_train_model[column] = X_train_model[column].fillna("missing").astype(str)
    X_validation_model[column] = X_validation_model[column].fillna("missing").astype(str)

print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric/binary features: {len(numeric_features)}")
print(f"Total model features: {len(model_features)}")


Categorical features: 3
Numeric/binary features: 9
Total model features: 12


## 6. Preprocessing


In [6]:
numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)


## 7. Lineare Regression


In [7]:
linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_regression_pipeline.fit(X_train_model, y_train)
linear_regression_predictions = linear_regression_pipeline.predict(X_validation_model)

linear_regression_mae = mean_absolute_error(
    y_validation, linear_regression_predictions
)
linear_regression_rmse = np.sqrt(
    mean_squared_error(y_validation, linear_regression_predictions)
)
linear_regression_r2 = r2_score(y_validation, linear_regression_predictions)

print(f"Validation MAE: {linear_regression_mae:.3f}")
print(f"Validation RMSE: {linear_regression_rmse:.3f}")
print(f"Validation R2: {linear_regression_r2:.3f}")


Validation MAE: 6.271
Validation RMSE: 10.214
Validation R2: 0.017


## 8. Generalized Additive Model (GAM)

Das Generalized Additive Model (GAM) erweitert die lineare Regression um nichtlineare, geglättete Funktionen einzelner Einflussfaktoren. Dadurch können beispielsweise nichtlineare Zusammenhänge zwischen Temperatur, Wind oder Niederschlag und der Verspätung modelliert werden, ohne die additive und vergleichsweise gut nachvollziehbare Modellstruktur aufzugeben. Kategoriale Einflussgrößen werden als Faktor-Terme und kontinuierliche Einflussgrößen als Spline-Terme berücksichtigt. Interaktionsterme werden zunächst nicht modelliert.


In [8]:
X_train_gam = X_train[model_features].copy()
X_validation_gam = X_validation[model_features].copy()

gam_factor_features: list[str] = [
    "event_hour",
    "event_weekday",
    "rr_precipitation_form",
]

gam_linear_features: list[str] = [
    "is_public_holiday",
    "rr_precipitation_indicator",
]

gam_spline_features: list[str] = [
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
]

gam_category_mappings: dict[str, dict[str, int]] = {}

for column in gam_factor_features:
    train_values = X_train_gam[column].fillna("missing").astype(str)
    validation_values = X_validation_gam[column].fillna("missing").astype(str)

    categories = sorted(train_values.unique().tolist())
    if "missing" not in categories:
        categories.append("missing")

    category_mapping = {category: code for code, category in enumerate(categories)}
    gam_category_mappings[column] = category_mapping

    known_categories = set(category_mapping)
    X_train_gam[column] = train_values.map(category_mapping).astype(int)
    X_validation_gam[column] = (
        validation_values.where(validation_values.isin(known_categories), "missing")
        .map(category_mapping)
        .astype(int)
    )

gam_numeric_features: list[str] = gam_linear_features + gam_spline_features
gam_numeric_imputer = SimpleImputer(strategy="median")
X_train_gam[gam_numeric_features] = gam_numeric_imputer.fit_transform(
    X_train_gam[gam_numeric_features]
)
X_validation_gam[gam_numeric_features] = gam_numeric_imputer.transform(
    X_validation_gam[gam_numeric_features]
)

X_train_gam = X_train_gam[model_features]
X_validation_gam = X_validation_gam[model_features]


def build_gam_terms(n_splines: int, spline_lam: float):
    gam_terms = None

    for feature_index, feature in enumerate(model_features):
        if feature in gam_factor_features:
            term = f(int(feature_index))
        elif feature in gam_linear_features:
            term = l(int(feature_index))
        elif feature in gam_spline_features:
            term = s(int(feature_index), n_splines=int(n_splines), lam=float(spline_lam))
        else:
            raise ValueError(f"Unexpected GAM feature: {feature}")

        gam_terms = term if gam_terms is None else gam_terms + term

    return gam_terms


def build_gam(n_splines: int, spline_lam: float) -> LinearGAM:
    return LinearGAM(build_gam_terms(n_splines=n_splines, spline_lam=spline_lam))


gam_model = build_gam(n_splines=10, spline_lam=1.0)
gam_model.fit(X_train_gam, y_train)
gam_predictions = gam_model.predict(X_validation_gam)

gam_mae = mean_absolute_error(y_validation, gam_predictions)
gam_rmse = np.sqrt(mean_squared_error(y_validation, gam_predictions))
gam_r2 = r2_score(y_validation, gam_predictions)

print(f"Validation MAE: {gam_mae:.3f}")
print(f"Validation RMSE: {gam_rmse:.3f}")
print(f"Validation R2: {gam_r2:.3f}")


KeyboardInterrupt: 

## 9. Random Forest

Der Random Forest wird als nichtlineares baumbasiertes Verfahren eingesetzt, um mögliche nichtlineare Zusammenhänge und Interaktionen zwischen den externen Einflussfaktoren und der Zugverspätung abzubilden.


In [ ]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_pipeline.fit(X_train_model, y_train)
random_forest_predictions = random_forest_pipeline.predict(X_validation_model)

random_forest_mae = mean_absolute_error(y_validation, random_forest_predictions)
random_forest_rmse = np.sqrt(
    mean_squared_error(y_validation, random_forest_predictions)
)
random_forest_r2 = r2_score(y_validation, random_forest_predictions)

print(f"Validation MAE: {random_forest_mae:.3f}")
print(f"Validation RMSE: {random_forest_rmse:.3f}")
print(f"Validation R2: {random_forest_r2:.3f}")


## 10. XGBoost

XGBoost wird als Gradient-Boosting-Verfahren eingesetzt, um nichtlineare Zusammenhänge und Interaktionen zwischen den externen Einflussfaktoren und der Zugverspätung abzubilden. Im Unterschied zum Random Forest werden die Entscheidungsbäume sequenziell aufgebaut, sodass nachfolgende Bäume insbesondere die Fehler der vorherigen Bäume adressieren.

`tree_method="hist"` wird verwendet, da der Datensatz relativ groß ist und diese Variante effizienter trainiert.


In [ ]:
xgboost_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgboost_pipeline.fit(X_train_model, y_train)
xgboost_predictions = xgboost_pipeline.predict(X_validation_model)

xgboost_mae = mean_absolute_error(y_validation, xgboost_predictions)
xgboost_rmse = np.sqrt(mean_squared_error(y_validation, xgboost_predictions))
xgboost_r2 = r2_score(y_validation, xgboost_predictions)

print(f"Validation MAE: {xgboost_mae:.3f}")
print(f"Validation RMSE: {xgboost_rmse:.3f}")
print(f"Validation R2: {xgboost_r2:.3f}")


## 11. CatBoost

CatBoost wird als nichtlineares Gradient-Boosting-Verfahren eingesetzt. Im Unterschied zur linearen Regression, zum Random Forest und zu XGBoost kann CatBoost kategoriale Merkmale direkt verarbeiten, sodass für dieses Modell kein One-Hot-Encoding erforderlich ist.


In [11]:
X_train_catboost = X_train_model.copy()
X_validation_catboost = X_validation_model.copy()

catboost_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)

catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_features,
)
catboost_predictions = catboost_model.predict(X_validation_catboost)

catboost_mae = mean_absolute_error(y_validation, catboost_predictions)
catboost_rmse = np.sqrt(mean_squared_error(y_validation, catboost_predictions))
catboost_r2 = r2_score(y_validation, catboost_predictions)

print(f"Validation MAE: {catboost_mae:.3f}")
print(f"Validation RMSE: {catboost_rmse:.3f}")
print(f"Validation R2: {catboost_r2:.3f}")


Validation MAE: 6.357
Validation RMSE: 10.223
Validation R2: 0.016


## 12. Modellvergleich


In [ ]:
model_comparison = pd.DataFrame(
    [
        {"Model": "Median Baseline", "MAE": baseline_mae, "RMSE": baseline_rmse, "R2": baseline_r2},
        {
            "Model": "Mean Baseline",
            "MAE": mean_baseline_mae,
            "RMSE": mean_baseline_rmse,
            "R2": mean_baseline_r2,
        },
        {
            "Model": "Linear Regression",
            "MAE": linear_regression_mae,
            "RMSE": linear_regression_rmse,
            "R2": linear_regression_r2,
        },
        {
            "Model": "GAM",
            "MAE": gam_mae,
            "RMSE": gam_rmse,
            "R2": gam_r2,
        },
        {
            "Model": "Random Forest",
            "MAE": random_forest_mae,
            "RMSE": random_forest_rmse,
            "R2": random_forest_r2,
        },
        {
            "Model": "XGBoost",
            "MAE": xgboost_mae,
            "RMSE": xgboost_rmse,
            "R2": xgboost_r2,
        },
        {"Model": "CatBoost", "MAE": catboost_mae, "RMSE": catboost_rmse, "R2": catboost_r2},
    ]
)

model_comparison.round(3)


## 13. Training vs. Validation

Zur Beurteilung einer möglichen Überanpassung werden die bereits trainierten Modelle zusätzlich auf den Trainingsdaten November 2025 bis Juni 2026 ausgewertet. Ein deutlicher Unterschied zwischen Trainings- und Validierungsleistung kann auf Overfitting bzw. eine eingeschränkte zeitliche Generalisierbarkeit hinweisen. Beide Splits sind nach der aktualisierten Boundary-Exclusion-Logik voneinander getrennt.

In [ ]:
linear_regression_train_predictions = linear_regression_pipeline.predict(X_train_model)
linear_regression_train_mae = mean_absolute_error(y_train, linear_regression_train_predictions)
linear_regression_train_rmse = np.sqrt(mean_squared_error(y_train, linear_regression_train_predictions))
linear_regression_train_r2 = r2_score(y_train, linear_regression_train_predictions)

gam_train_predictions = gam_model.predict(X_train_gam)
gam_train_mae = mean_absolute_error(y_train, gam_train_predictions)
gam_train_rmse = np.sqrt(mean_squared_error(y_train, gam_train_predictions))
gam_train_r2 = r2_score(y_train, gam_train_predictions)

random_forest_train_predictions = random_forest_pipeline.predict(X_train_model)
random_forest_train_mae = mean_absolute_error(y_train, random_forest_train_predictions)
random_forest_train_rmse = np.sqrt(mean_squared_error(y_train, random_forest_train_predictions))
random_forest_train_r2 = r2_score(y_train, random_forest_train_predictions)

xgboost_train_predictions = xgboost_pipeline.predict(X_train_model)
xgboost_train_mae = mean_absolute_error(y_train, xgboost_train_predictions)
xgboost_train_rmse = np.sqrt(mean_squared_error(y_train, xgboost_train_predictions))
xgboost_train_r2 = r2_score(y_train, xgboost_train_predictions)

catboost_train_predictions = catboost_model.predict(X_train_catboost)
catboost_train_mae = mean_absolute_error(y_train, catboost_train_predictions)
catboost_train_rmse = np.sqrt(mean_squared_error(y_train, catboost_train_predictions))
catboost_train_r2 = r2_score(y_train, catboost_train_predictions)

training_validation_comparison = pd.DataFrame(
    [
        {
            "Model": "Linear Regression",
            "Dataset": "Training",
            "MAE": linear_regression_train_mae,
            "RMSE": linear_regression_train_rmse,
            "R2": linear_regression_train_r2,
        },
        {
            "Model": "Linear Regression",
            "Dataset": "Validation",
            "MAE": linear_regression_mae,
            "RMSE": linear_regression_rmse,
            "R2": linear_regression_r2,
        },
        {
            "Model": "GAM",
            "Dataset": "Training",
            "MAE": gam_train_mae,
            "RMSE": gam_train_rmse,
            "R2": gam_train_r2,
        },
        {
            "Model": "GAM",
            "Dataset": "Validation",
            "MAE": gam_mae,
            "RMSE": gam_rmse,
            "R2": gam_r2,
        },
        {
            "Model": "Random Forest",
            "Dataset": "Training",
            "MAE": random_forest_train_mae,
            "RMSE": random_forest_train_rmse,
            "R2": random_forest_train_r2,
        },
        {
            "Model": "Random Forest",
            "Dataset": "Validation",
            "MAE": random_forest_mae,
            "RMSE": random_forest_rmse,
            "R2": random_forest_r2,
        },
        {
            "Model": "XGBoost",
            "Dataset": "Training",
            "MAE": xgboost_train_mae,
            "RMSE": xgboost_train_rmse,
            "R2": xgboost_train_r2,
        },
        {
            "Model": "XGBoost",
            "Dataset": "Validation",
            "MAE": xgboost_mae,
            "RMSE": xgboost_rmse,
            "R2": xgboost_r2,
        },
        {
            "Model": "CatBoost",
            "Dataset": "Training",
            "MAE": catboost_train_mae,
            "RMSE": catboost_train_rmse,
            "R2": catboost_train_r2,
        },
        {
            "Model": "CatBoost",
            "Dataset": "Validation",
            "MAE": catboost_mae,
            "RMSE": catboost_rmse,
            "R2": catboost_r2,
        },
    ]
)

training_validation_comparison.round(3)


### R²-Gap zwischen Training und Validation


In [ ]:
r2_gap = pd.DataFrame(
    [
        {
            "Model": "Linear Regression",
            "Train R2": linear_regression_train_r2,
            "Validation R2": linear_regression_r2,
            "R2 Gap": linear_regression_train_r2 - linear_regression_r2,
        },
        {
            "Model": "GAM",
            "Train R2": gam_train_r2,
            "Validation R2": gam_r2,
            "R2 Gap": gam_train_r2 - gam_r2,
        },
        {
            "Model": "Random Forest",
            "Train R2": random_forest_train_r2,
            "Validation R2": random_forest_r2,
            "R2 Gap": random_forest_train_r2 - random_forest_r2,
        },
        {
            "Model": "XGBoost",
            "Train R2": xgboost_train_r2,
            "Validation R2": xgboost_r2,
            "R2 Gap": xgboost_train_r2 - xgboost_r2,
        },
        {
            "Model": "CatBoost",
            "Train R2": catboost_train_r2,
            "Validation R2": catboost_r2,
            "R2 Gap": catboost_train_r2 - catboost_r2,
        },
    ]
)

r2_gap.round(3)


## 14. Robustheitsanalyse: logarithmische Transformation der Zielvariable

Aufgrund der stark rechtsschiefen Verteilung der Verspätungen und einzelner extremer Verspätungswerte wird ergänzend geprüft, ob eine logarithmische Transformation der Zielvariable die Prognosegüte verbessert. Hierfür wird `log1p(delay_target)` verwendet. Die Vorhersagen werden anschließend mit `expm1()` zurück auf die ursprüngliche Minutenskala transformiert, sodass MAE, RMSE und R² weiterhin in Bezug auf die ursprüngliche Zielvariable berechnet werden können.

Die log1p-Transformation wird als ergänzende Robustheitsanalyse betrachtet. Da R² die zentrale Vergleichskennzahl der Hauptanalyse ist, wird eine Log-Variante nur dann als Hauptmodellierungsansatz berücksichtigt, wenn sie auch auf der ursprünglichen Minutenskala eine bessere R²-Leistung zeigt. MAE und RMSE bleiben ergänzende Kennzahlen.


In [ ]:
y_train_log = np.log1p(y_train)

linear_regression_log_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", LinearRegression())]
)
linear_regression_log_pipeline.fit(X_train_model, y_train_log)
linear_regression_log_predictions = linear_regression_log_pipeline.predict(X_validation_model)
linear_regression_log_predictions = np.maximum(np.expm1(linear_regression_log_predictions), 0)
linear_regression_log_mae = mean_absolute_error(y_validation, linear_regression_log_predictions)
linear_regression_log_rmse = np.sqrt(mean_squared_error(y_validation, linear_regression_log_predictions))
linear_regression_log_r2 = r2_score(y_validation, linear_regression_log_predictions)

gam_log_model = build_gam(n_splines=10, spline_lam=1.0)
gam_log_model.fit(X_train_gam, y_train_log)
gam_log_predictions = gam_log_model.predict(X_validation_gam)
gam_log_predictions = np.expm1(gam_log_predictions)
gam_log_predictions = np.maximum(gam_log_predictions, 0)
gam_log_mae = mean_absolute_error(y_validation, gam_log_predictions)
gam_log_rmse = np.sqrt(mean_squared_error(y_validation, gam_log_predictions))
gam_log_r2 = r2_score(y_validation, gam_log_predictions)

random_forest_log_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        ),
    ]
)
random_forest_log_pipeline.fit(X_train_model, y_train_log)
random_forest_log_predictions = random_forest_log_pipeline.predict(X_validation_model)
random_forest_log_predictions = np.maximum(np.expm1(random_forest_log_predictions), 0)
random_forest_log_mae = mean_absolute_error(y_validation, random_forest_log_predictions)
random_forest_log_rmse = np.sqrt(mean_squared_error(y_validation, random_forest_log_predictions))
random_forest_log_r2 = r2_score(y_validation, random_forest_log_predictions)

xgboost_log_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)
xgboost_log_pipeline.fit(X_train_model, y_train_log)
xgboost_log_predictions = xgboost_log_pipeline.predict(X_validation_model)
xgboost_log_predictions = np.expm1(xgboost_log_predictions)
xgboost_log_predictions = np.maximum(xgboost_log_predictions, 0)
xgboost_log_mae = mean_absolute_error(y_validation, xgboost_log_predictions)
xgboost_log_rmse = np.sqrt(mean_squared_error(y_validation, xgboost_log_predictions))
xgboost_log_r2 = r2_score(y_validation, xgboost_log_predictions)

catboost_log_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)
catboost_log_model.fit(X_train_catboost, y_train_log, cat_features=categorical_features)
catboost_log_predictions = catboost_log_model.predict(X_validation_catboost)
catboost_log_predictions = np.maximum(np.expm1(catboost_log_predictions), 0)
catboost_log_mae = mean_absolute_error(y_validation, catboost_log_predictions)
catboost_log_rmse = np.sqrt(mean_squared_error(y_validation, catboost_log_predictions))
catboost_log_r2 = r2_score(y_validation, catboost_log_predictions)

print("Linear Regression (log1p):")
print(f"MAE {linear_regression_log_mae:.3f}")
print(f"RMSE {linear_regression_log_rmse:.3f}")
print(f"R2 {linear_regression_log_r2:.3f}")
print()
print("GAM (log1p):")
print(f"MAE {gam_log_mae:.3f}")
print(f"RMSE {gam_log_rmse:.3f}")
print(f"R2 {gam_log_r2:.3f}")
print()
print("Random Forest (log1p):")
print(f"MAE {random_forest_log_mae:.3f}")
print(f"RMSE {random_forest_log_rmse:.3f}")
print(f"R2 {random_forest_log_r2:.3f}")
print()
print("XGBoost (log1p):")
print(f"MAE {xgboost_log_mae:.3f}")
print(f"RMSE {xgboost_log_rmse:.3f}")
print(f"R2 {xgboost_log_r2:.3f}")
print()
print("CatBoost (log1p):")
print(f"MAE {catboost_log_mae:.3f}")
print(f"RMSE {catboost_log_rmse:.3f}")
print(f"R2 {catboost_log_r2:.3f}")


## 15. Ergebnisse der Robustheitsanalyse


In [ ]:
model_comparison_with_log_target = pd.DataFrame(
    [
        {"Model": "Median Baseline", "MAE": baseline_mae, "RMSE": baseline_rmse, "R2": baseline_r2},
        {
            "Model": "Mean Baseline",
            "MAE": mean_baseline_mae,
            "RMSE": mean_baseline_rmse,
            "R2": mean_baseline_r2,
        },
        {
            "Model": "Linear Regression",
            "MAE": linear_regression_mae,
            "RMSE": linear_regression_rmse,
            "R2": linear_regression_r2,
        },
        {
            "Model": "GAM",
            "MAE": gam_mae,
            "RMSE": gam_rmse,
            "R2": gam_r2,
        },
        {
            "Model": "Random Forest",
            "MAE": random_forest_mae,
            "RMSE": random_forest_rmse,
            "R2": random_forest_r2,
        },
        {
            "Model": "XGBoost",
            "MAE": xgboost_mae,
            "RMSE": xgboost_rmse,
            "R2": xgboost_r2,
        },
        {"Model": "CatBoost", "MAE": catboost_mae, "RMSE": catboost_rmse, "R2": catboost_r2},
        {
            "Model": "Linear Regression (log1p)",
            "MAE": linear_regression_log_mae,
            "RMSE": linear_regression_log_rmse,
            "R2": linear_regression_log_r2,
        },
        {
            "Model": "GAM (log1p)",
            "MAE": gam_log_mae,
            "RMSE": gam_log_rmse,
            "R2": gam_log_r2,
        },
        {
            "Model": "Random Forest (log1p)",
            "MAE": random_forest_log_mae,
            "RMSE": random_forest_log_rmse,
            "R2": random_forest_log_r2,
        },
        {
            "Model": "XGBoost (log1p)",
            "MAE": xgboost_log_mae,
            "RMSE": xgboost_log_rmse,
            "R2": xgboost_log_r2,
        },
        {
            "Model": "CatBoost (log1p)",
            "MAE": catboost_log_mae,
            "RMSE": catboost_log_rmse,
            "R2": catboost_log_r2,
        },
    ]
)

model_comparison_with_log_target.round(3)


## 16. Hyperparameter-Tuning GAM

Beim GAM werden ausschließlich die Flexibilität der Spline-Funktionen und deren Glättungsstärke in einem kleinen, vorab begrenzten Raster variiert. Die Hyperparametersuche bleibt bewusst klein, um eine übermäßige Anpassung an den Validierungsmonat Juli zu vermeiden.


In [ ]:
gam_param_grid = [
    {"n_splines": 8, "spline_lam": 0.6},
    {"n_splines": 8, "spline_lam": 5.0},
    {"n_splines": 12, "spline_lam": 0.6},
    {"n_splines": 12, "spline_lam": 5.0},
]

gam_tuning_rows = []

for params in gam_param_grid:
    model = build_gam(**params)
    model.fit(X_train_gam, y_train)
    train_predictions = model.predict(X_train_gam)
    validation_predictions = model.predict(X_validation_gam)
    gam_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation R2": r2_score(y_validation, validation_predictions),
            "Validation MAE": mean_absolute_error(y_validation, validation_predictions),
            "Validation RMSE": np.sqrt(mean_squared_error(y_validation, validation_predictions)),
        }
    )

gam_tuning_results = pd.DataFrame(gam_tuning_rows).sort_values(
    "Validation R2",
    ascending=False,
).reset_index(drop=True)

best_gam_params = gam_tuning_results.loc[
    0,
    ["n_splines", "spline_lam"],
].to_dict()
best_gam_params["n_splines"] = int(best_gam_params["n_splines"])

gam_tuned_model = build_gam(**best_gam_params)
gam_tuned_model.fit(X_train_gam, y_train)
gam_tuned_train_predictions = gam_tuned_model.predict(X_train_gam)
gam_tuned_predictions = gam_tuned_model.predict(X_validation_gam)

gam_tuned_train_r2 = r2_score(y_train, gam_tuned_train_predictions)
gam_tuned_mae = mean_absolute_error(y_validation, gam_tuned_predictions)
gam_tuned_rmse = np.sqrt(mean_squared_error(y_validation, gam_tuned_predictions))
gam_tuned_r2 = r2_score(y_validation, gam_tuned_predictions)

gam_tuning_results.round(3)


## 17. Hyperparameter-Tuning Random Forest

Der Random Forest zeigt eine deutlich bessere Trainings- als Validierungsleistung, was auf Overfitting hindeutet. Deshalb werden wenige zentrale Hyperparameter gezielt angepasst, um die Generalisierung auf den Validierungsdatensatz zu verbessern.

Die Hyperparametersuche wird bewusst auf wenige vorab festgelegte Kombinationen begrenzt, um die Modellentwicklung nachvollziehbar zu halten und eine übermäßige Anpassung an den Validierungsmonat Juli zu vermeiden.


In [ ]:
random_forest_param_grid = [
    {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 5},
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 10},
    {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 5},
    {"n_estimators": 300, "max_depth": 15, "min_samples_leaf": 10},
]

random_forest_tuning_rows = []

for params in random_forest_param_grid:
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    **params,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    pipeline.fit(X_train_model, y_train)
    train_predictions = pipeline.predict(X_train_model)
    validation_predictions = pipeline.predict(X_validation_model)
    random_forest_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation R2": r2_score(y_validation, validation_predictions),
            "Validation MAE": mean_absolute_error(y_validation, validation_predictions),
            "Validation RMSE": np.sqrt(mean_squared_error(y_validation, validation_predictions)),
        }
    )

random_forest_tuning_results = pd.DataFrame(random_forest_tuning_rows).sort_values(
    "Validation R2",
    ascending=False,
).reset_index(drop=True)

best_random_forest_params = random_forest_tuning_results.loc[
    0,
    ["n_estimators", "max_depth", "min_samples_leaf"],
].to_dict()
best_random_forest_params = {
    key: int(value) for key, value in best_random_forest_params.items()
}

random_forest_tuned_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                **best_random_forest_params,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_tuned_pipeline.fit(X_train_model, y_train)
random_forest_tuned_train_predictions = random_forest_tuned_pipeline.predict(X_train_model)
random_forest_tuned_predictions = random_forest_tuned_pipeline.predict(X_validation_model)

random_forest_tuned_train_r2 = r2_score(y_train, random_forest_tuned_train_predictions)
random_forest_tuned_mae = mean_absolute_error(y_validation, random_forest_tuned_predictions)
random_forest_tuned_rmse = np.sqrt(mean_squared_error(y_validation, random_forest_tuned_predictions))
random_forest_tuned_r2 = r2_score(y_validation, random_forest_tuned_predictions)

random_forest_tuning_results.round(3)


## 18. Hyperparameter-Tuning XGBoost

Die Hyperparametersuche wird bewusst auf wenige vorab festgelegte Kombinationen begrenzt, um die Modellentwicklung nachvollziehbar zu halten und eine übermäßige Anpassung an den Validierungsmonat Juli zu vermeiden.


In [ ]:
xgboost_param_grid = [
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.03},
    {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03},
    {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.03},
]

xgboost_tuning_rows = []

for params in xgboost_param_grid:
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                XGBRegressor(
                    **params,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    pipeline.fit(X_train_model, y_train)
    train_predictions = pipeline.predict(X_train_model)
    validation_predictions = pipeline.predict(X_validation_model)
    xgboost_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation R2": r2_score(y_validation, validation_predictions),
            "Validation MAE": mean_absolute_error(y_validation, validation_predictions),
            "Validation RMSE": np.sqrt(mean_squared_error(y_validation, validation_predictions)),
        }
    )

xgboost_tuning_results = pd.DataFrame(xgboost_tuning_rows).sort_values(
    "Validation R2",
    ascending=False,
).reset_index(drop=True)

best_xgboost_params = xgboost_tuning_results.loc[
    0,
    ["n_estimators", "max_depth", "learning_rate"],
].to_dict()
best_xgboost_params["n_estimators"] = int(best_xgboost_params["n_estimators"])
best_xgboost_params["max_depth"] = int(best_xgboost_params["max_depth"])

xgboost_tuned_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                **best_xgboost_params,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgboost_tuned_pipeline.fit(X_train_model, y_train)
xgboost_tuned_train_predictions = xgboost_tuned_pipeline.predict(X_train_model)
xgboost_tuned_predictions = xgboost_tuned_pipeline.predict(X_validation_model)

xgboost_tuned_train_r2 = r2_score(y_train, xgboost_tuned_train_predictions)
xgboost_tuned_mae = mean_absolute_error(y_validation, xgboost_tuned_predictions)
xgboost_tuned_rmse = np.sqrt(mean_squared_error(y_validation, xgboost_tuned_predictions))
xgboost_tuned_r2 = r2_score(y_validation, xgboost_tuned_predictions)

xgboost_tuning_results.round(3)


## 19. Hyperparameter-Tuning CatBoost

Für CatBoost wird ebenfalls eine kleine, gezielte Auswahl zentraler Hyperparameter geprüft. Ziel ist eine Verbesserung der Validierungsleistung ohne umfangreiche Hyperparameteroptimierung.

Die Hyperparametersuche wird bewusst auf wenige vorab festgelegte Kombinationen begrenzt, um die Modellentwicklung nachvollziehbar zu halten und eine übermäßige Anpassung an den Validierungsmonat Juli zu vermeiden.


In [12]:
catboost_param_grid = [
    # 1. Iteration
    {"iterations": 200, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3},
    {"iterations": 400, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3},

    # 2. Iteration
    {"iterations": 300, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},

    # 3. Iteration
    {"iterations": 250, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 350, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 250, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 350, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.025, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.04, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 7},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 15},
    {"iterations": 300, "depth": 5, "learning_rate": 0.04, "l2_leaf_reg": 10},

    # 4. Iteration
    {"iterations": 300, "depth": 6, "learning_rate": 0.0150, "l2_leaf_reg": 10},
    {"iterations": 350, "depth": 6, "learning_rate": 0.0150, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 5, "learning_rate": 0.0200, "l2_leaf_reg": 10},

    {"iterations": 350, "depth": 6, "learning_rate": 0.0125, "l2_leaf_reg": 10},
    {"iterations": 400, "depth": 6, "learning_rate": 0.0125, "l2_leaf_reg": 10},

    {"iterations": 400, "depth": 6, "learning_rate": 0.0100, "l2_leaf_reg": 10},
    {"iterations": 450, "depth": 6, "learning_rate": 0.0100, "l2_leaf_reg": 10},
    {"iterations": 500, "depth": 6, "learning_rate": 0.0100, "l2_leaf_reg": 10},

    {"iterations": 500, "depth": 6, "learning_rate": 0.0075, "l2_leaf_reg": 10},
    {"iterations": 600, "depth": 6, "learning_rate": 0.0075, "l2_leaf_reg": 10},

    {"iterations": 800, "depth": 6, "learning_rate": 0.0050, "l2_leaf_reg": 10},
]

catboost_train_pool = Pool(
    X_train_catboost,
    y_train,
    cat_features=categorical_features,
)
catboost_validation_pool = Pool(
    X_validation_catboost,
    y_validation,
    cat_features=categorical_features,
)

catboost_tuning_rows = []
catboost_tuned_model = None
best_catboost_validation_r2 = -np.inf

for params in catboost_param_grid:
    model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
    )
    model.fit(catboost_train_pool)

    train_predictions = model.predict(catboost_train_pool)
    validation_predictions = model.predict(catboost_validation_pool)
    validation_r2 = r2_score(y_validation, validation_predictions)

    catboost_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation R2": validation_r2,
            "Validation MAE": mean_absolute_error(y_validation, validation_predictions),
            "Validation RMSE": np.sqrt(mean_squared_error(y_validation, validation_predictions)),
        }
    )

    if validation_r2 > best_catboost_validation_r2:
        best_catboost_validation_r2 = validation_r2
        catboost_tuned_model = model
        catboost_tuned_train_predictions = train_predictions
        catboost_tuned_predictions = validation_predictions

catboost_tuning_results = pd.DataFrame(catboost_tuning_rows).sort_values(
    "Validation R2",
    ascending=False,
).reset_index(drop=True)

best_catboost_params = catboost_tuning_results.loc[
    0,
    ["iterations", "depth", "learning_rate", "l2_leaf_reg"],
].to_dict()
best_catboost_params["iterations"] = int(best_catboost_params["iterations"])
best_catboost_params["depth"] = int(best_catboost_params["depth"])
best_catboost_params["learning_rate"] = float(best_catboost_params["learning_rate"])
best_catboost_params["l2_leaf_reg"] = float(best_catboost_params["l2_leaf_reg"])

catboost_tuned_train_r2 = r2_score(y_train, catboost_tuned_train_predictions)
catboost_tuned_mae = mean_absolute_error(y_validation, catboost_tuned_predictions)
catboost_tuned_rmse = np.sqrt(mean_squared_error(y_validation, catboost_tuned_predictions))
catboost_tuned_r2 = r2_score(y_validation, catboost_tuned_predictions)

catboost_tuning_results.round(
    {
        "learning_rate": 4,
        "l2_leaf_reg": 3,
        "Train R2": 6,
        "Validation R2": 6,
        "Validation MAE": 3,
        "Validation RMSE": 3,
    }
)


,iterations,depth,learning_rate,l2_leaf_reg,Train R2,Validation R2,Validation MAE,Validation RMSE
0,300,6,0.0150,10,0.041116,0.018819,6.315,10.206
1,350,6,0.0150,10,0.043076,0.018531,6.320,10.207
2,600,6,0.0075,10,0.041054,0.018262,6.319,10.209
3,800,6,0.0050,10,0.039624,0.018258,6.311,10.209
4,500,6,0.0075,10,0.038642,0.018204,6.315,10.209
5,300,5,0.0200,10,0.040177,0.018127,6.308,10.209
6,400,6,0.0100,10,0.039608,0.018111,6.307,10.210
7,350,6,0.0125,10,0.040559,0.017923,6.310,10.211
8,450,6,0.0100,10,0.041142,0.017909,6.306,10.211
9,400,6,0.0125,10,0.042397,0.017906,6.314,10.211


## 20. Vergleich nach Tuning


In [ ]:
model_comparison_tuned = pd.DataFrame(
    [
        {"Model": "Median Baseline", "MAE": baseline_mae, "RMSE": baseline_rmse, "R2": baseline_r2},
        {
            "Model": "Mean Baseline",
            "MAE": mean_baseline_mae,
            "RMSE": mean_baseline_rmse,
            "R2": mean_baseline_r2,
        },
        {
            "Model": "Linear Regression",
            "MAE": linear_regression_mae,
            "RMSE": linear_regression_rmse,
            "R2": linear_regression_r2,
        },
        {
            "Model": "GAM",
            "MAE": gam_mae,
            "RMSE": gam_rmse,
            "R2": gam_r2,
        },
        {
            "Model": "GAM Tuned",
            "MAE": gam_tuned_mae,
            "RMSE": gam_tuned_rmse,
            "R2": gam_tuned_r2,
        },
        {
            "Model": "Random Forest",
            "MAE": random_forest_mae,
            "RMSE": random_forest_rmse,
            "R2": random_forest_r2,
        },
        {
            "Model": "Random Forest Tuned",
            "MAE": random_forest_tuned_mae,
            "RMSE": random_forest_tuned_rmse,
            "R2": random_forest_tuned_r2,
        },
        {
            "Model": "XGBoost",
            "MAE": xgboost_mae,
            "RMSE": xgboost_rmse,
            "R2": xgboost_r2,
        },
        {
            "Model": "XGBoost Tuned",
            "MAE": xgboost_tuned_mae,
            "RMSE": xgboost_tuned_rmse,
            "R2": xgboost_tuned_r2,
        },
        {"Model": "CatBoost", "MAE": catboost_mae, "RMSE": catboost_rmse, "R2": catboost_r2},
        {
            "Model": "CatBoost Tuned",
            "MAE": catboost_tuned_mae,
            "RMSE": catboost_tuned_rmse,
            "R2": catboost_tuned_r2,
        },
    ]
)

model_comparison_tuned.round(3)


## 21. Zwischenfazit der Modellauswahl

Die Modellauswahl basiert primär auf dem R² des Validierungsdatensatzes Juli 2026. MAE und RMSE werden ergänzend berücksichtigt. Die logarithmische Transformation der Zielvariable wird als Robustheitsanalyse behandelt. Das Hyperparameter-Tuning wurde bewusst auf wenige Kombinationen begrenzt. Der Testdatensatz August 2026 bleibt weiterhin vollständig unangetastet.


In [ ]:
final_validation_overview = pd.DataFrame(
    [
        {"Model": "Median Baseline", "MAE": baseline_mae, "RMSE": baseline_rmse, "R2": baseline_r2},
        {
            "Model": "Mean Baseline",
            "MAE": mean_baseline_mae,
            "RMSE": mean_baseline_rmse,
            "R2": mean_baseline_r2,
        },
        {
            "Model": "Linear Regression",
            "MAE": linear_regression_mae,
            "RMSE": linear_regression_rmse,
            "R2": linear_regression_r2,
        },
        {
            "Model": "GAM",
            "MAE": gam_mae,
            "RMSE": gam_rmse,
            "R2": gam_r2,
        },
        {
            "Model": "GAM Tuned",
            "MAE": gam_tuned_mae,
            "RMSE": gam_tuned_rmse,
            "R2": gam_tuned_r2,
        },
        {
            "Model": "Random Forest",
            "MAE": random_forest_mae,
            "RMSE": random_forest_rmse,
            "R2": random_forest_r2,
        },
        {
            "Model": "Random Forest Tuned",
            "MAE": random_forest_tuned_mae,
            "RMSE": random_forest_tuned_rmse,
            "R2": random_forest_tuned_r2,
        },
        {
            "Model": "XGBoost",
            "MAE": xgboost_mae,
            "RMSE": xgboost_rmse,
            "R2": xgboost_r2,
        },
        {
            "Model": "XGBoost Tuned",
            "MAE": xgboost_tuned_mae,
            "RMSE": xgboost_tuned_rmse,
            "R2": xgboost_tuned_r2,
        },
        {"Model": "CatBoost", "MAE": catboost_mae, "RMSE": catboost_rmse, "R2": catboost_r2},
        {
            "Model": "CatBoost Tuned",
            "MAE": catboost_tuned_mae,
            "RMSE": catboost_tuned_rmse,
            "R2": catboost_tuned_r2,
        },
    ]
)

final_validation_overview.round(3)
